In [0]:
# DDL GENERATOR UTILITY
def generate_ddl(
    metadata_df,
    catalog,
    schema,
    table_name
):

    columns = metadata_df.select(
        "column_name",
        "sql_type"
    ).collect()

    column_definitions = []

    for row in columns:

        column_name = row["column_name"]

        sql_type = row["sql_type"]

        column_definitions.append(
            f"{column_name} {sql_type}"
        )

    ddl = f"""
    CREATE TABLE IF NOT EXISTS
    {catalog}.{schema}.{table_name}
    (
        {', '.join(column_definitions)}
    )
    USING DELTA
    """

    return ddl

In [0]:
# TABLE CREATOR UTILITY

def create_table(ddl):

    spark.sql(ddl)

# ============================================
# TABLE EXISTS
# ============================================

def table_exists(
    catalog,
    schema,
    table_name
):

    full_table_name = (
        f"{catalog}.{schema}.{table_name}"
    )

    return spark.catalog.tableExists(
        full_table_name
    )


# ============================================
# SCHEMA EVOLUTION
# ============================================

def evolve_table_schema(
    metadata_df,
    target_table
):

    existing_columns = {

        field.name.lower()

        for field in spark.table(
            target_table
        ).schema.fields

    }

    metadata_rows = metadata_df.collect()

    for row in metadata_rows:

        column_name = row["column_name"]

        data_type = row["sql_type"]

        if column_name.lower() not in existing_columns:

            print(
                f"Adding New Column: "
                f"{column_name}"
            )

            spark.sql(
                f"""
                ALTER TABLE {target_table}
                ADD COLUMN
                (
                    {column_name}
                    {data_type}
                )
                """
            )